In [41]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler , LabelEncoder
import os

In [42]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("uciml/breast-cancer-wisconsin-data")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'breast-cancer-wisconsin-data' dataset.
Path to dataset files: /kaggle/input/breast-cancer-wisconsin-data


In [43]:
print(path)

/kaggle/input/breast-cancer-wisconsin-data


In [44]:
df = pd.read_csv(os.path.join(path, "data.csv"))

In [45]:
df.head(5)

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,radius_se,texture_se,perimeter_se,area_se,smoothness_se,compactness_se,concavity_se,concave points_se,symmetry_se,fractal_dimension_se,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,0.4956,1.1560,3.445,27.23,0.009110,0.07458,0.05661,0.01867,0.05963,0.009208,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.011490,0.02461,0.05688,0.01885,0.01756,0.005115,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [46]:
df.shape

(569, 33)

In [47]:
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)

In [48]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

In [49]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [50]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [51]:
X_train_tensor = torch.from_numpy(X_train).float()
X_test_tensor = torch.from_numpy(X_test).float()
y_train_tensor = torch.from_numpy(y_train).float()
y_test_tensor = torch.from_numpy(y_test).float()

In [52]:
X_train_tensor.shape , X_test_tensor.shape , y_train_tensor.shape , y_test_tensor.shape

(torch.Size([455, 30]),
 torch.Size([114, 30]),
 torch.Size([455]),
 torch.Size([114]))

In [53]:
class MySimpleNN():

  def __init__(self , X) -> None:
    self.weights = torch.randn(X.shape[1],1,dtype=torch.float32 , requires_grad=True)
    self.bias = torch.zeros(1,dtype=torch.float32 , requires_grad=True)

  def forward(self , X):
    z = torch.matmul(X , self.weights) + self.bias
    y_pred = torch.sigmoid(z)
    return y_pred

  def loss(self , y_pred , y):
    epsilon = 1e-7
    y_pred = torch.clamp(y_pred, epsilon, 1.0 - epsilon)
    loss = -(y_train_tensor * torch.log(y_pred) + (1 - y_train_tensor) * torch.log(1-y_pred)).mean()
    return loss


In [54]:
learnign_rate = 0.1
epochs = 50

In [55]:
# Create a model
model = MySimpleNN(X_train_tensor)

# define a loop
for epoch in range(epochs):
  y_pred = model.forward(X_train_tensor)  # Forward pass

  # Loss calculation
  loss = model.loss(y_pred , y_train_tensor)

  # Backward pass
  loss.backward()

  # parameters update
  with torch.no_grad():
    model.weights -= learnign_rate * model.weights.grad
    model.bias -= learnign_rate * model.bias.grad

  # zero gradients
  model.weights.grad.zero_()
  model.bias.grad.zero_()

  # print loss in each epoch
  print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

Epoch: 1, Loss: 2.547884941101074
Epoch: 2, Loss: 2.421779155731201
Epoch: 3, Loss: 2.301703691482544
Epoch: 4, Loss: 2.185716390609741
Epoch: 5, Loss: 2.0762863159179688
Epoch: 6, Loss: 1.9743468761444092
Epoch: 7, Loss: 1.87907075881958
Epoch: 8, Loss: 1.7919836044311523
Epoch: 9, Loss: 1.7127383947372437
Epoch: 10, Loss: 1.6397498846054077
Epoch: 11, Loss: 1.5740375518798828
Epoch: 12, Loss: 1.5145139694213867
Epoch: 13, Loss: 1.461802363395691
Epoch: 14, Loss: 1.4153015613555908
Epoch: 15, Loss: 1.3743479251861572
Epoch: 16, Loss: 1.338243842124939
Epoch: 17, Loss: 1.3060121536254883
Epoch: 18, Loss: 1.2765908241271973
Epoch: 19, Loss: 1.2502903938293457
Epoch: 20, Loss: 1.2265896797180176
Epoch: 21, Loss: 1.2050511837005615
Epoch: 22, Loss: 1.185316562652588
Epoch: 23, Loss: 1.1670970916748047
Epoch: 24, Loss: 1.1501636505126953
Epoch: 25, Loss: 1.1343367099761963
Epoch: 26, Loss: 1.1194769144058228
Epoch: 27, Loss: 1.105475664138794
Epoch: 28, Loss: 1.092247486114502
Epoch: 29, L

In [56]:
# model evaluation
with torch.no_grad():
  y_pred = model.forward(X_test_tensor)
  y_pred = (y_pred > 0.9).float()
  accuracy = (y_pred == y_test_tensor).float().mean()
  print(f'Accuracy: {accuracy.item()}')


Accuracy: 0.59602952003479
